In [13]:
from pathlib import Path

from oauthlib.oauth2 import BackendApplicationClient
from rasterio.transform import from_bounds
from requests_oauthlib import OAuth2Session
from rasterio.crs import CRS as RioCRS

import io
import rasterio
import numpy as np

In [17]:
QUARTERS = [
    ("01-01", "03-31"),
    ("04-01", "06-30"),
    ("07-01", "09-30"),
    ("10-01", "12-31"),
]

CLIENT_ID     = "NOT_SET"
CLIENT_SECRET = "NOT_SET"

AOI_BBOX = {
    "west":  18.50000,  # ←
    "south": 48.77910,  # ↓
    "east":  18.83029,  # →
    "north": 49.01070,  # ↑
}

OUTPUT_DIR    = Path("./_output")
RESOLUTION_M  = 10




CLIENT_ID = ""
CLIENT_SECRET = ""


OUTPUT_DIR    = Path("./_output")

In [3]:
def sentinelhub_compliance_hook(response):
    response.raise_for_status()
    return response

In [21]:
client = BackendApplicationClient(client_id=CLIENT_ID)
oauth = OAuth2Session(client=client)
oauth.register_compliance_hook("access_token_response", sentinelhub_compliance_hook)

# Get token for the session
token = oauth.fetch_token(token_url='https://identity.dataspace.copernicus.eu/auth/realms/CDSE/protocol/openid-connect/token',
                          client_secret=CLIENT_SECRET, include_client_id=True)

# All requests using this session will have an access token automatically added
resp = oauth.get("https://sh.dataspace.copernicus.eu/configuration/v1/wms/instances")
print(resp.content)

b'<html>\n<head>\n<meta http-equiv="Content-Type" content="text/html;charset=utf-8"/>\n<title>Error 500 Request failed.</title>\n</head>\n<body><h2>HTTP ERROR 500 Request failed.</h2>\n<table>\n<tr><th>URI:</th><td>/configuration/v1/wms/instances</td></tr>\n<tr><th>STATUS:</th><td>500</td></tr>\n<tr><th>MESSAGE:</th><td>Request failed.</td></tr>\n<tr><th>SERVLET:</th><td>org.glassfish.jersey.servlet.ServletContainer-3fe98084</td></tr>\n</table>\n\n</body>\n</html>\n'


In [31]:
evalscript = """
//VERSION=3
function setup() {
  return {
    input: [ "B04", "B08", "observations", "dataMask"],
    output: { bands: 4,
              sampleType: "FLOAT32" // Preserves decimals for calculations
            }
  };
}

function evaluatePixel(sample) {
  return [sample.B04/10000, sample.B08/10000, sample.observations, sample.dataMask];
}
"""

request = {
  "input": {
    "bounds": {
      "bbox": [
        18.50000,
        48.77910,
        18.83029,
        49.01070
      ]
    },
    "data": [
      {
        "dataFilter": {
          "timeRange": {
            "from": "2020-01-01T00:00:00Z",
            "to": "2020-01-02T23:59:59Z"
          }
        },
        "type": "byoc-5460de54-082e-473a-b6ea-d5cbe3c17cca"
      }
    ]
  },
  "output": {
    "width": 2500,
    "height": 2500,
    "responses": [
      {
        "identifier": "default",
        "format": {
          "type": "image/tiff"
        }
      }
    ]
  },
  "evalscript": evalscript,
}


In [32]:
url = "https://sh.dataspace.copernicus.eu/process/v1"
response = oauth.post(url, json=request)
response.raise_for_status()

In [29]:
def save_geotiff(pixels, bbox, out_path, no_data=-32768):
    rows, cols = pixels.shape
    transform = from_bounds(
        bbox["west"], bbox["south"], bbox["east"], bbox["north"],
        cols, rows
    )
    out_path.parent.mkdir(parents=True, exist_ok=True)

    with rasterio.open(
            out_path,
            mode = "w",
            driver="GTiff",
            height=rows,
            width=cols,
            count=1,
            dtype=np.float32,
            crs=RioCRS.from_epsg(4326),
            transform=transform,
            nodata=no_data,
            compress="lzw",
            predctor=3,
            tiled=True,
    ) as dst:
        dst.write(pixels.astype(np.float32), 1)
        dst.update_tags(
            DESCRIPTION="Sentinel-2 Quarterly Cloudless Mosaic NDVI",
            COLLECTION_ID="byoc-5460de54-082e-473a-b6ea-d5cbe3c17cca",
            RESOLUTION_M=str(RESOLUTION_M),
        )

    size_mb = out_path.stat().st_size / 1e6
    print(f"  → Saved: {out_path}  ({cols}×{rows} px, {size_mb:.1f} MB)")

In [30]:
bytes_io = io.BytesIO(response.content)

band_names = [ "B04", "B08", "observations", "dataMask"]

with rasterio.open(bytes_io) as src:
    for i, band in enumerate(band_names):
        band_data = np.array(src.read(i+1))

        save_path = OUTPUT_DIR / f"Oauth_{band}_2020_Q1.tif"
        save_geotiff(band_data, AOI_BBOX, save_path)

  → Saved: _output/Oauth_B04_2020_Q1.tif  (2500×2500 px, 16.3 MB)
  → Saved: _output/Oauth_B08_2020_Q1.tif  (2500×2500 px, 17.4 MB)
  → Saved: _output/Oauth_observations_2020_Q1.tif  (2500×2500 px, 1.5 MB)
  → Saved: _output/Oauth_dataMask_2020_Q1.tif  (2500×2500 px, 0.2 MB)
